# 27 Fitting: formula fits with each search method, weighted fits, other lines, bends, powers, dose-response curves

Real-data checks of the words in the group "Fitting" that notebooks 01 to 17 leave open. This notebook holds only
simplydata: it reads the data with `sdata.table`, runs the words, reads the numbers back with simplydata's own words
(`get_column`, `get_row`, `get_cell_at`, through `Result`) and compares them with the reference values in
`references/27_fitting.csv`. Those come from `scripts/references/27_fitting.py`, which runs the reference programs
directly (R, statsmodels, SciPy, the algorithms written out in NumPy) and holds the certified values; it also saves
the R datasets this notebook reads (`references/data/27_fitting/`, made at run time, not kept in the repository).

| section | word and option | data | certified / published values |
|---|---|---|---|
| 1 | `fit_formula(formula, start=, test_method="trust region" / "dogbox" / "levenberg-marquardt")` | NIST StRD nonlinear regression (26 problems, NIST start 2) | NIST certified values |
| 2 | `fit_curve(of_degree=)`; `fit_curve(weights=)`; `fit_formula(weights=)` | NIST Pontius; cars; Puromycin | NIST certified values |
| 3 | `fit_line(test_method="passing-bablok" / "siegel")` | serum and plasma creatinine (mcr) | none (Passing & Bablok 1983 and Siegel 1982 algorithms written out) |
| 4 | `fit_line(with_bends=1)` | stagnant band height (segmented; Bacon & Watts 1971) | none (R segmented) |
| 5 | `find_best_power_of(test_method="box-cox" / "yeo-johnson")` | poison survival times (Box & Cox 1964) | none here (notebook 14) |
| 6 | `fit_dose_response(of_kind=, doses_for_effect=, outcome="yes or no", out_of=)` | ryegrass, selenium (drc) | none (R drc help page examples) |

Tolerances (set in the reference file, one per kind of calculation): NIST certified values by the log relative
error, at least 6 digits for nonlinear problems and 9 for linear ones; relative 1e-10 for closed formulas and least
squares; relative 1e-6 for fits by iteration.

In [1]:
import simplydata as sdata

from reference_checks import Comparisons, Result, quiet

cmp = Comparisons("27_fitting")
DATA = "../data/"
FETCHED = "../data/fetched/"  # the datasets this repository does not host (written by scripts/get_data.py)

## 1. The three search methods of `fit_formula` on NIST's nonlinear problems

Every problem is fitted from NIST's second start with `test_method="trust region"`, `"dogbox"` and
`"levenberg-marquardt"`; each estimate, standard error and the residual standard deviation is compared with NIST's
certified value. A method that stops early is reported as it is (FAIL).

In [2]:
RATIONAL3 = "(b1 + b2*x + b3*x**2 + b4*x**3) / (1 + b5*x + b6*x**2 + b7*x**3)"
NIST_FORMULAS = {
    "Misra1a": "y = b1*(1-exp(-b2*x))", "Chwirut2": "y = exp(-b1*x)/(b2+b3*x)", "Chwirut1": "y = exp(-b1*x)/(b2+b3*x)",
    "Lanczos3": "y = b1*exp(-b2*x) + b3*exp(-b4*x) + b5*exp(-b6*x)",
    "Gauss1": "y = b1*exp(-b2*x) + b3*exp(-(x-b4)**2/b5**2) + b6*exp(-(x-b7)**2/b8**2)",
    "Gauss2": "y = b1*exp(-b2*x) + b3*exp(-(x-b4)**2/b5**2) + b6*exp(-(x-b7)**2/b8**2)",
    "DanWood": "y = b1*x**b2", "Misra1b": "y = b1*(1-(1+b2*x/2)**(-2))",
    "Kirby2": "y = (b1 + b2*x + b3*x**2) / (1 + b4*x + b5*x**2)", "Hahn1": "y = " + RATIONAL3,
    "MGH17": "y = b1 + b2*exp(-x*b4) + b3*exp(-x*b5)", "Lanczos1": "y = b1*exp(-b2*x) + b3*exp(-b4*x) + b5*exp(-b6*x)",
    "Lanczos2": "y = b1*exp(-b2*x) + b3*exp(-b4*x) + b5*exp(-b6*x)",
    "Gauss3": "y = b1*exp(-b2*x) + b3*exp(-(x-b4)**2/b5**2) + b6*exp(-(x-b7)**2/b8**2)",
    "Misra1c": "y = b1*(1-(1+2*b2*x)**(-0.5))", "Misra1d": "y = b1*b2*x*((1+b2*x)**(-1))",
    "Roszman1": "y = b1 - b2*x - arctan(b3/(x-b4))/pi",
    "ENSO": "y = b1 + b2*cos(2*pi*x/12) + b3*sin(2*pi*x/12) + b5*cos(2*pi*x/b4) + b6*sin(2*pi*x/b4)"
            " + b8*cos(2*pi*x/b7) + b9*sin(2*pi*x/b7)",
    "MGH09": "y = b1*(x**2+x*b2) / (x**2+x*b3+b4)", "Thurber": "y = " + RATIONAL3, "BoxBOD": "y = b1*(1-exp(-b2*x))",
    "Rat42": "y = b1 / (1+exp(b2-b3*x))", "MGH10": "y = b1 * exp(b2/(x+b3))",
    "Eckerle4": "y = (b1/b2) * exp(-0.5*((x-b3)/b2)**2)", "Rat43": "y = b1 / ((1+exp(b2-b3*x))**(1/b4))",
    "Bennett5": "y = b1 * (b2+x)**(-1/b3)",
}
starts = Result(sdata.table(cmp.data_file("nist_starts")))
START = {}
for problem, parameter, value in zip(starts["problem"], starts["parameter"], starts["start 2"]):
    START.setdefault(problem, {})[parameter] = value
sdata.table(cmp.data_file("nist_Misra1a")).fit_formula(NIST_FORMULAS["Misra1a"], start=START["Misra1a"], test_method="trust region")

Read "nist_starts.csv" with "," between the columns (guessed). To be sure, give separator=",".
Read "nist_Misra1a.csv" with "," between the columns (guessed). To be sure, give separator=",".
Fitted "y = b1*(1-exp(-b2*x))" to 14 rows: b1 = 239 (95 % CI 233 to 245), b2 = 0.000550 (95 % CI 0.000534 to 0.000566). Residual standard error 0.102 on 12 degrees of freedom; R squared = 1.00 (only a rough guide for a curve).
Values found: b1, b2. Data: x. Method: nonlinear least squares (trust region reflective), as R nls(); intervals are estimate +/- t x standard error (R confint() on nls profiles the likelihood and can differ a little).


,test,formula,b1,standard error of b1,t of b1,p-value of b1,95 % CI low of b1,95 % CI high of b1,b2,standard error of b2,...,95 % CI low of b2,95 % CI high of b2,residual standard error,degrees of freedom,R squared,AIC,rows,converged,method,note
1,formula fit (nonlinear least squares),y = b1*(1-exp(-b2*x)),238.942129,2.707008,88.267996,2.985633e-18,233.044066,244.840192,0.00055,0.000007,...,0.000534,0.000566,0.101879,12.0,0.999982,-20.37904,14,yes,trust region reflective,converged


In [3]:
for name, formula in NIST_FORMULAS.items():
    data = sdata.table(cmp.data_file(f"nist_{name}"))
    for test_method in ("trust region", "dogbox", "levenberg-marquardt"):
        method = f'fit_formula (start 2, test_method="{test_method}")'
        try:
            fit = Result(quiet(data.fit_formula, formula, start=START[name], test_method=test_method)).row(1)
        except Exception as problem:  # noqa: BLE001 - a fit that stops with an error is recorded, not hidden
            cmp.skip(f"NIST {name}", method, "all values", "NIST certified", f"stopped: {str(problem)[:150]}")
            continue
        for p in START[name]:
            cmp.check(f"NIST {name}", method, f"{p} estimate", fit[p])
            cmp.check(f"NIST {name}", method, f"{p} standard error", fit[f"standard error of {p}"])
        cmp.check(f"NIST {name}", method, "residual standard deviation", fit["residual standard error"])

Read "nist_Misra1a.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  NIST Misra1a | fit_formula (start 2, test_method="trust region") | b1 estimate: simplydata 238.94212918, NIST StRD certified values 238.94212918; LRE 11.7 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1a.shtml
agrees  NIST Misra1a | fit_formula (start 2, test_method="trust region") | b1 standard error: simplydata 2.7070075241, NIST StRD certified values 2.7070075241; LRE 11.7 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1a.shtml
agrees  NIST Misra1a | fit_formula (start 2, test_method="trust region") | b2 estimate: simplydata 0.000550156431802, NIST StRD certified values 0.00055015643181; LRE 10.8 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1a.shtml
agrees  NIST Mis

agrees  NIST Misra1b | fit_formula (start 2, test_method="trust region") | b1 estimate: simplydata 337.997458171, NIST StRD certified values 337.99746163; LRE 8.0 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1b.shtml
agrees  NIST Misra1b | fit_formula (start 2, test_method="trust region") | b1 standard error: simplydata 3.16439495209, NIST StRD certified values 3.1643950207; LRE 7.7 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1b.shtml
agrees  NIST Misra1b | fit_formula (start 2, test_method="trust region") | b2 estimate: simplydata 0.000390390917517, NIST StRD certified values 0.00039039091287; LRE 7.9 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1b.shtml
agrees  NIST Misra1b | fit_formula (start 2, test_method="trust region") | b2 standard error: simplydata 4.254732

agrees  NIST Kirby2 | fit_formula (start 2, test_method="trust region") | b1 estimate: simplydata 1.67450630643, NIST StRD certified values 1.6745063063; LRE 10.1 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/kirby2.shtml
agrees  NIST Kirby2 | fit_formula (start 2, test_method="trust region") | b1 standard error: simplydata 0.0879896343631, NIST StRD certified values 0.087989634338; LRE 9.5 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/kirby2.shtml
agrees  NIST Kirby2 | fit_formula (start 2, test_method="trust region") | b2 estimate: simplydata -0.139273978681, NIST StRD certified values -0.13927397867; LRE 10.1 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/kirby2.shtml
agrees  NIST Kirby2 | fit_formula (start 2, test_method="trust region") | b2 standard error: simplydata 0.00411820414

agrees  NIST Misra1c | fit_formula (start 2, test_method="trust region") | b1 estimate: simplydata 636.427258087, NIST StRD certified values 636.42725809; LRE 11.3 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1c.shtml
agrees  NIST Misra1c | fit_formula (start 2, test_method="trust region") | b1 standard error: simplydata 4.66383265897, NIST StRD certified values 4.6638326572; LRE 9.4 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1c.shtml
agrees  NIST Misra1c | fit_formula (start 2, test_method="trust region") | b2 estimate: simplydata 0.000208136272566, NIST StRD certified values 0.00020813627256; LRE 10.6 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1c.shtml
agrees  NIST Misra1c | fit_formula (start 2, test_method="trust region") | b2 standard error: simplydata 1.7728

agrees  NIST Misra1d | fit_formula (start 2, test_method="dogbox") | b1 estimate: simplydata 437.369707542, NIST StRD certified values 437.36970754; LRE 11.4 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1d.shtml
agrees  NIST Misra1d | fit_formula (start 2, test_method="dogbox") | b1 standard error: simplydata 3.64891743448, NIST StRD certified values 3.6489174345; LRE 11.2 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1d.shtml
agrees  NIST Misra1d | fit_formula (start 2, test_method="dogbox") | b2 estimate: simplydata 0.000302273244489, NIST StRD certified values 0.00030227324449; LRE 11.4 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/misra1d.shtml
agrees  NIST Misra1d | fit_formula (start 2, test_method="dogbox") | b2 standard error: simplydata 2.93343544789e-06, NIST StRD 

Read "nist_Thurber.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  NIST Thurber | fit_formula (start 2, test_method="trust region") | b1 estimate: simplydata 1288.13968002, NIST StRD certified values 1288.13968; LRE 10.9 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/thurber.shtml
agrees  NIST Thurber | fit_formula (start 2, test_method="trust region") | b1 standard error: simplydata 4.66479633442, NIST StRD certified values 4.6647963344; LRE 11.3 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/thurber.shtml
agrees  NIST Thurber | fit_formula (start 2, test_method="trust region") | b2 estimate: simplydata 1491.0792525, NIST StRD certified values 1491.0792535; LRE 9.2 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/thurber.shtml
agrees  NIST Thurber | fit_formula (start 2, test_method="trust region") | b2 standard error: simplydata 39.5711565255, NI

agrees  NIST Eckerle4 | fit_formula (start 2, test_method="levenberg-marquardt") | b1 estimate: simplydata 1.55438271801, NIST StRD certified values 1.5543827178; LRE 9.9 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/eckerle4.shtml
agrees  NIST Eckerle4 | fit_formula (start 2, test_method="levenberg-marquardt") | b1 standard error: simplydata 0.0154080512311, NIST StRD certified values 0.015408051163; LRE 8.4 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/eckerle4.shtml
agrees  NIST Eckerle4 | fit_formula (start 2, test_method="levenberg-marquardt") | b2 estimate: simplydata 4.08883217694, NIST StRD certified values 4.0888321754; LRE 9.4 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/eckerle4.shtml
agrees  NIST Eckerle4 | fit_formula (start 2, test_method="levenberg-marquardt") | b2 stan

agrees  NIST Bennett5 | fit_formula (start 2, test_method="trust region") | b1 estimate: simplydata -2523.50581701, NIST StRD certified values -2523.5058043; LRE 8.3 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/bennett5.shtml
agrees  NIST Bennett5 | fit_formula (start 2, test_method="trust region") | b1 standard error: simplydata 297.151759304, NIST StRD certified values 297.15175411; LRE 7.8 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/bennett5.shtml
agrees  NIST Bennett5 | fit_formula (start 2, test_method="trust region") | b2 estimate: simplydata 46.7365646968, NIST StRD certified values 46.736564644; LRE 8.9 (needs >= 6)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/nls/data/bennett5.shtml
agrees  NIST Bennett5 | fit_formula (start 2, test_method="trust region") | b2 standard error: simplydata 1.24488

## 2. Polynomial and weighted fits

`fit_curve(of_degree=2)` on NIST Pontius (certified values, at least 9 digits); `fit_curve(of_degree=2,
weights="speed")` on cars (R `lm(dist ~ poly(speed, 2, raw = TRUE), weights = speed)`, statsmodels `WLS`);
`fit_formula(weights="conc")`: the Michaelis-Menten curve on the treated Puromycin rows (R `nls(weights =)`, SciPy
`curve_fit(sigma = 1/sqrt(w))`).

In [4]:
pc = Result(quiet(sdata.table(cmp.data_file("nist_Pontius")).fit_curve, x="x", y="y", of_degree=2), key="term")
for term in ["(intercept)", "x", "x**2"]:
    cmp.check("NIST Pontius", "fit_curve (of_degree=2)", f"{term} estimate", pc[term, "estimate"])
    cmp.check("NIST Pontius", "fit_curve (of_degree=2)", f"{term} standard error", pc[term, "standard error"])

method = 'fit_curve (of_degree=2, weights="speed")'
wc = Result(quiet(sdata.table(cmp.data_file("cars")).fit_curve, x="speed", y="dist", of_degree=2, weights="speed"), key="term")
for term in ["(intercept)", "speed", "speed**2"]:
    for quantity in ("estimate", "standard error", "p-value"):
        cmp.check("cars", method, f"{term} {quantity}", wc[term, quantity])

method = 'fit_formula (weights="conc")'
pf = Result(quiet(sdata.table(DATA + "puromycin.csv").fit_formula, "rate = Vm * conc / (K + conc)", start={"Vm": 200, "K": 0.1},
                  weights="conc")).row(1)
cmp.check_all("Puromycin (treated)", method, {"Vm": pf["Vm"], "K": pf["K"], "Vm standard error": pf["standard error of Vm"],
                                              "K standard error": pf["standard error of K"],
                                              "residual standard error": pf["residual standard error"]})

Read "nist_Pontius.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  NIST Pontius | fit_curve (of_degree=2) | (intercept) estimate: simplydata 0.000673565789473, NIST StRD certified values 0.000673565789474; LRE 12.2 (needs >= 9)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Pontius.shtml
agrees  NIST Pontius | fit_curve (of_degree=2) | (intercept) standard error: simplydata 0.000107938612033, NIST StRD certified values 0.000107938612033; LRE 12.1 (needs >= 9)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Pontius.shtml
agrees  NIST Pontius | fit_curve (of_degree=2) | x estimate: simplydata 7.32059160401e-07, NIST StRD certified values 7.32059160401e-07; LRE 14.9 (needs >= 9)
      Source: NIST StRD certified values (certified) - https://www.itl.nist.gov/div898/strd/lls/data/Pontius.shtml
agrees  NIST Pontius | fit_curve (of_degree=2) | x stand

## 3. Lines for comparing two measurement methods: Passing-Bablok and Siegel

The serum and plasma creatinine of 108 patients (R mcr's data, in `data/`): Passing-Bablok (R `mcr::mcreg(method.reg
= "PaBa", method.ci = "analytical")`, which simplydata calls, and the 1983 algorithm written out in NumPy) and the
Siegel repeated-medians line (SciPy `siegelslopes(method="hierarchical")` and the repeated medians written out).

In [5]:
creatinine = sdata.table(FETCHED + "creatinine.csv", separator=",")
pb = Result(quiet(creatinine.fit_line, x="serum", y="plasma", test_method="passing-bablok")).row(1)
cmp.check_all("creatinine (mcr)", 'fit_line (test_method="passing-bablok")',
              {"slope": pb["slope"], "slope 95 % low": pb["slope 95 % CI low"], "slope 95 % high": pb["slope 95 % CI high"],
               "intercept": pb["intercept"], "intercept 95 % low": pb["intercept 95 % CI low"],
               "intercept 95 % high": pb["intercept 95 % CI high"]})
sg = Result(quiet(creatinine.fit_line, x="serum", y="plasma", test_method="siegel")).row(1)
cmp.check_all("creatinine (mcr)", 'fit_line (test_method="siegel")', {"slope": sg["slope"], "intercept": sg["intercept"]})

agrees  creatinine (mcr) | fit_line (test_method="passing-bablok") | slope: simplydata 1.08800890272, R mcr mcreg (PaBa, analytical), help-page data (mcr 1.3.3.1) 1.08800890272; relative difference 0.00e+00 (allowed 1e-10)
      Source: R mcr mcreg (PaBa, analytical), help-page data (mcr 1.3.3.1) (run reference) - https://search.r-project.org/CRAN/refmans/mcr/html/mcreg.html
agrees  creatinine (mcr) | fit_line (test_method="passing-bablok") | slope 95 % low: simplydata 1, R mcr mcreg (PaBa, analytical), help-page data (mcr 1.3.3.1) 1; relative difference 2.22e-16 (allowed 1e-10)
      Source: R mcr mcreg (PaBa, analytical), help-page data (mcr 1.3.3.1) (run reference) - https://search.r-project.org/CRAN/refmans/mcr/html/mcreg.html
agrees  creatinine (mcr) | fit_line (test_method="passing-bablok") | slope 95 % high: simplydata 1.17300462437, R mcr mcreg (PaBa, analytical), help-page data (mcr 1.3.3.1) 1.17300462437; relative difference 1.89e-16 (allowed 1e-10)
      Source: R mcr mcreg 

## 4. A line with a bend: the stagnant band data

Bacon & Watts' (1971) stagnant surface layer data (segmented): `fit_line(x="x", y="y", with_bends=1)` against R
`segmented::segmented(lm(y ~ x), seg.Z = ~ x)` with Davies' test.

In [6]:
sb = Result(quiet(sdata.table(cmp.data_file("stagnant")).fit_line, x="x", y="y", with_bends=1)).row(1)
cmp.check_all("stagnant (segmented)", "fit_line (with_bends=1)",
              {k: sb[k] for k in ["bend 1", "bend 1 95 % CI low", "bend 1 95 % CI high", "slope 1", "slope 1 standard error", "slope 2",
                                  "slope 2 standard error", "intercept", "Davies p-value", "residual standard error"]})

Read "stagnant.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  stagnant (segmented) | fit_line (with_bends=1) | bend 1: simplydata 0.041105784554, R segmented segmented + davies.test (segmented 2.3.0) 0.0411057839274; relative difference 1.52e-08 (allowed 1e-06)
      Source: R segmented segmented + davies.test (segmented 2.3.0) (run reference) - https://search.r-project.org/CRAN/refmans/segmented/html/segmented.html
agrees  stagnant (segmented) | fit_line (with_bends=1) | bend 1 95 % CI low: simplydata -0.00602294028792, R segmented segmented + davies.test (segmented 2.3.0) -0.00602294092364; relative difference 1.06e-07 (allowed 1e-06)
      Source: R segmented segmented + davies.test (segmented 2.3.0) (run reference) - https://search.r-project.org/CRAN/refmans/segmented/html/segmented.html
agrees  stagnant (segmented) | fit_line (with_bends=1) | bend 1 95 % CI high: simplydata 0.0882345093958, R segmented segmented + davies.test (segmented 2.3.0)

## 5. The best power: Box-Cox and Yeo-Johnson on the poison survival times

`find_best_power_of("time")` (Box-Cox, with its profile 95 % interval) and `test_method="yeo-johnson"`, against R
`car::powerTransform` and the likelihoods maximised in SciPy.

In [7]:
poisons = sdata.table(DATA + "poisons.csv")
bc = Result(quiet(poisons.find_best_power_of, "time")).row(1)
cmp.check_all("poisons (Box & Cox 1964)", 'find_best_power_of (test_method="box-cox")',
              {"power": bc["power"], "95 % low": bc["95 % CI low"], "95 % high": bc["95 % CI high"]})
yj = Result(quiet(poisons.find_best_power_of, "time", test_method="yeo-johnson")).row(1)
cmp.check("poisons (Box & Cox 1964)", 'find_best_power_of (test_method="yeo-johnson")', "power", yj["power"])

Read "poisons.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  poisons (Box & Cox 1964) | find_best_power_of (test_method="box-cox") | power: simplydata -0.360158442079, SciPy stats.boxcox (scipy 1.18.1) -0.360158442079; relative difference 0.00e+00 (allowed 1e-06)
      Source: SciPy stats.boxcox (scipy 1.18.1) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.boxcox.html
agrees  poisons (Box & Cox 1964) | find_best_power_of (test_method="box-cox") | 95 % low: simplydata -0.961993862358, SciPy stats.boxcox (scipy 1.18.1) -0.961993862358; relative difference 0.00e+00 (allowed 1e-06)
      Source: SciPy stats.boxcox (scipy 1.18.1) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.boxcox.html
agrees  poisons (Box & Cox 1964) | find_best_power_of (test_method="box-cox") | 95 % high: simplydata 0.226643264044, SciPy stats.boxcox (scipy 1.18.1) 0.226643264044; relative differe

1

## 6. Dose-response curves: the drc help-page examples

Ryegrass: `of_kind="3 parameter log-logistic"`, `"5 parameter log-logistic"` and `"weibull"`, each with
`doses_for_effect=[10, 50, 90]`; selenium (type 1): `outcome="yes or no", out_of="total"`. References: drc with its
default stopping rule (the call simplydata makes) and with `relTol = 1e-12` (FINDINGS A7).

In [8]:
ryegrass = sdata.table(FETCHED + "ryegrass.csv", separator=",")
PARAMETERS = {"3 parameter log-logistic": ["b", "d", "e"], "5 parameter log-logistic": ["b", "c", "d", "e", "f"],
              "weibull": ["b", "c", "d", "e"]}
for kind, params in PARAMETERS.items():
    method = f'fit_dose_response (of_kind="{kind}", doses_for_effect=[10, 50, 90])'
    fit = Result(quiet(ryegrass.fit_dose_response, x="conc", y="root_length", of_kind=kind, doses_for_effect=[10, 50, 90]), key="term")
    for p in params:
        term = [t for t in fit.index if t.startswith(f"{p} (")][0]
        cmp.check("ryegrass (drc)", method, f"{p} estimate", fit[term, "estimate"])
        cmp.check("ryegrass (drc)", method, f"{p} standard error", fit[term, "standard error"])
    for ed in ("ED10", "ED50", "ED90"):
        cmp.check_all("ryegrass (drc)", method, {ed: fit[ed, "estimate"], f"{ed} standard error": fit[ed, "standard error"],
                                                 f"{ed} 95 % low": fit[ed, "95 % CI low"], f"{ed} 95 % high": fit[ed, "95 % CI high"]})

fit = Result(quiet(sdata.table(cmp.data_file("selenium_type1")).fit_dose_response, x="conc", y="dead", outcome="yes or no",
                   out_of="total"), key="term")
cmp.check_all("selenium, type 1 (drc)", 'fit_dose_response (outcome="yes or no", out_of="total")',
              {"b estimate": fit["b (steepness)", "estimate"], "e estimate": fit["e (ED50)", "estimate"],
               "b standard error": fit["b (steepness)", "standard error"], "e standard error": fit["e (ED50)", "standard error"],
               "ED50 95 % low": fit["ED50", "95 % CI low"], "ED50 95 % high": fit["ED50", "95 % CI high"]})

agrees  ryegrass (drc) | fit_dose_response (of_kind="3 parameter log-logistic", doses_for_effect=[10, 50, 90]) | b estimate: simplydata 2.47032669235, R drc drm + ED (help-page ryegrass example) (drc 4.0.0) 2.47032679727; relative difference 4.25e-08 (allowed 1e-06)
      Source: R drc drm + ED (help-page ryegrass example) (drc 4.0.0) (run reference) - https://search.r-project.org/CRAN/refmans/drc/html/ED.drc.html
agrees  ryegrass (drc) | fit_dose_response (of_kind="3 parameter log-logistic", doses_for_effect=[10, 50, 90]) | b estimate: simplydata 2.47032669235, R drc drm + ED (relTol 1e-12) (drc 4.0.0) 2.47032657741; relative difference 4.65e-08 (allowed 1e-06)
      Source: R drc drm + ED (relTol 1e-12) (drc 4.0.0) (run reference) - https://search.r-project.org/CRAN/refmans/drc/html/ED.drc.html
agrees  ryegrass (drc) | fit_dose_response (of_kind="3 parameter log-logistic", doses_for_effect=[10, 50, 90]) | b standard error: simplydata 0.341683547716, R drc drm + ED (help-page ryegrass

## 7. The confidence band of `chart.fit_line(with_confidence_band=True)`

The band drawn around the least-squares line of plasma on serum creatinine, read back with `chart.get_numbers_drawn()`
(the low and high end at each of the 200 drawn x), against R `predict.lm(interval = "confidence")` at the same x, by
the largest relative difference.

In [9]:
creatinine = sdata.table(FETCHED + "creatinine.csv", separator=",")
chart = creatinine.start_chart()
quiet(chart.draw_dots, x="serum", y="plasma")
quiet(chart.fit_line, with_confidence_band=True)
band = [point for point in Result(quiet(chart.get_numbers_drawn)).columns_of(["kind", "low", "high"]) if point[0] == "band"]
cmp.check("creatinine (mcr)", "chart fit_line (band=True)", "the band drawn (R predict.lm, interval = confidence)",
          [point[1] for point in band] + [point[2] for point in band])

agrees  creatinine (mcr) | chart fit_line (band=True) | the band drawn (R predict.lm, interval = confidence): largest relative difference over 400 values: simplydata 8.0967989112e-16, R stats predict.lm (interval = confidence) (stats 4.6.2) 0; difference 8.10e-16 (allowed 1e-10)
      Source: R stats predict.lm (interval = confidence) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/predict.lm.html


1

In [10]:
cmp.finish()

27_fitting: 976 comparisons - 863 agrees, 3 explained difference – machine, 110 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,27_fitting,NIST Misra1a,"fit_formula (start 2, test_method=""trust region"")",b1 estimate,2.389421e+02,238.942129,NIST StRD certified values,4.396270e-10,6.000000e+00,lre,1.173521e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/nls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,27_fitting,NIST Misra1a,"fit_formula (start 2, test_method=""trust region"")",b1 standard error,2.707008e+00,2.707008,NIST StRD certified values,-4.884537e-12,6.000000e+00,lre,1.174367e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/nls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,27_fitting,NIST Misra1a,"fit_formula (start 2, test_method=""trust region"")",b2 estimate,5.501564e-04,0.000550,NIST StRD certified values,-8.222915e-15,6.000000e+00,lre,1.082546e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/nls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,27_fitting,NIST Misra1a,"fit_formula (start 2, test_method=""trust region"")",b2 standard error,7.266869e-06,0.000007,NIST StRD certified values,-1.508904e-16,6.000000e+00,lre,1.068269e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/nls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,27_fitting,NIST Misra1a,"fit_formula (start 2, test_method=""trust region"")",residual standard deviation,1.018788e-01,0.101879,NIST StRD certified values,2.429584e-12,6.000000e+00,lre,1.062255e+01,PASS,,agrees,,,,certified,https://www.itl.nist.gov/div898/strd/nls/data/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
971,27_fitting,"selenium, type 1 (drc)","fit_dose_response (outcome=""yes or no"", out_of...",b standard error,1.554740e-01,0.155474,"R drc drm (binomial), help-page selenium data ...",-2.668781e-08,1.000000e-06,relative,1.716545e-07,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/drc/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
972,27_fitting,"selenium, type 1 (drc)","fit_dose_response (outcome=""yes or no"", out_of...",e standard error,1.382686e+01,13.826860,"R drc drm (binomial), help-page selenium data ...",5.809424e-08,1.000000e-06,relative,4.201550e-09,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/drc/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
973,27_fitting,"selenium, type 1 (drc)","fit_dose_response (outcome=""yes or no"", out_of...",ED50 95 % low,2.251557e+02,225.155924,"R drc drm (binomial), help-page selenium data ...",-1.810280e-04,1.000000e-06,relative,8.040115e-07,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/drc/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
974,27_fitting,"selenium, type 1 (drc)","fit_dose_response (outcome=""yes or no"", out_of...",ED50 95 % high,2.793560e+02,279.356217,"R drc drm (binomial), help-page selenium data ...",-1.808002e-04,1.000000e-06,relative,6.472032e-07,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/drc/...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
